In [ ]:
import IPython as ipy
from arcgis.gis import GIS
from arcgis.features import Feature

import os
import pandas as pd

# Ensure that all interactive cell output is displayed (default="last_expr")
ipy.get_ipython().ast_node_interactivity = "all"

In [ ]:
# Login to your ArcGIS organization

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

ORG = os.getenv("ARCGIS_ORG_URL")
CID = os.getenv("ARCGIS_CLIENT_ID")
CSEC = os.getenv("ARCGIS_CLIENT_SECRET")
API_KEY = os.getenv("ARCGIS_API_KEY")

# --- Method A: Interactive OAuth (first run opens browser; later reuse profile) ---
# gis = GIS(ORG, client_id=CID, profile="org_oauth")
# After first authorization you can reconnect with:
# gis = GIS("org_oauth")

# --- Method B: OAuth client credentials (no user context; service level) ---
#gis = GIS(ORG, client_id=CID, client_secret=CSEC, grant_type="client_credentials")

# --- Method C: Username/password (stores a profile) ---
# gis = GIS(ORG, username=USR, password=PWD, profile="org_user")
# Later:
# gis = GIS("org_user")

# --- Method D: API Key (read-only / specific capabilities) ---
# gis = GIS(api_key=API_KEY)

# --- Method E: Using ArcGIS Pro login(in case you need to use Pro) ---
# gis = GIS("pro")

# --- Connect to ArcGIS Online using a built-in account ---    
# gis = GIS('home')

## Active login method: OAuth client credentials
gis = GIS(ORG, client_id=CID)
# gis  # <-- removed bare repr to prevent credential leakage in output

print("Successfully logged in as: " + gis.properties.user.username[:3] + "***")


In [ ]:
data = gis.content.get("3a1e92d564b44a4cb7ed549e79836e11")  # Replace with your item ID

df_incentivo = pd.DataFrame.spatial.from_layer(data.layers[0])

df_incentivo.head()


In [ ]:
df_incentivo.info()

In [ ]:
df_incentivo["FECHA_DE_LA_ACTIVIDAD"] = pd.to_datetime(df_incentivo["FECHA_DE_LA_ACTIVIDAD"], format="%Y-%m-%d")
df_incentivo.info()

In [ ]:
df_incentivo_montos = df_incentivo.filter(like="F202", axis="columns")
df_incentivo_montos.head()

In [ ]:
df_columns = df_incentivo.columns.sort_values()
df_columns

df_incentivo = df_incentivo[df_columns]



In [ ]:
df_incentivo["AREA_INCENTIVADA__pagada_"] = df_incentivo["ÁREA_INCENTIVADA__pagada_"]
df_incentivo = df_incentivo.drop("ÁREA_INCENTIVADA__pagada_", axis=1)



In [ ]:
df_incentivo.insert(df_incentivo.columns.get_loc("F2026___MONTO_DEL_INCENTIVO__QQ")+1, "AREA_INCENTIVADA__pagada_", df_incentivo.pop("AREA_INCENTIVADA__pagada_"))
df_incentivo.insert(0, "FECHA_DE_LA_ACTIVIDAD", df_incentivo.pop("FECHA_DE_LA_ACTIVIDAD"))

df_incentivo.head()
